# 1단계 EDA — 규칙 분포 확인 및 NER 태그 매핑 확정

**목표**: `data/` 아래 18,900개 판결문 JSON 전체를 훑어서

1. `rules_triggered`(R1~R10) 값이 실제로 몇 번, 어떤 텍스트에 등장하는지
2. `method` 값 종류 (`GENERALIZE`/`TOKENIZE`)
3. 분야(`caseClass`)별 문서 분포
4. span(`text[start:end]`)과 `original_text`가 정확히 일치하는 비율
5. 섹션 텍스트 길이 (BERT 512토큰 초과 비율)
6. 문서당 annotation 개수 분포

를 확인한다. 결과는 `계획.md`의 `RULE_TO_TAG` 매핑표를 확정하는 데 쓰인다.

> 이 노트북의 코드는 `scripts/eda_full_scan.py`와 같은 로직이며, 전체 18,900개 파일 스캔에 몇 분 걸리므로 결과를 캐시한 `scripts/eda_results.json`을 함께 읽어서 보여준다.

In [1]:
import json
import glob
from collections import Counter, defaultdict

BASE_TRAIN = "../data/12.판결서 익명처리 데이터/3.개방데이터/2.데이터(NIA)/Training/02.라벨링데이터"
BASE_VAL = "../data/12.판결서 익명처리 데이터/3.개방데이터/2.데이터(NIA)/Validation/02.라벨링데이터"

files = glob.glob(f"{BASE_TRAIN}/**/*.json", recursive=True) + glob.glob(f"{BASE_VAL}/**/*.json", recursive=True)
print(f"총 파일 수: {len(files)}")

In [2]:
# 실행 결과 (캐시): scripts/eda_results.json 참조

총 파일 수: 18900


## 1. 규칙(`rules_triggered`)별 분포

`rules_triggered`는 annotation 하나가 어떤 익명화 규칙(R1~R10)에 의해 만들어졌는지 나타낸다. 공식 문서(`official.md`)는 R1~R8이 실제 사용된다고 했고, R5·R6·R9·R10은 내용 미공개라고 했다. 전체 데이터를 세어보면 무엇이 실제로 등장하는지 알 수 있다.

In [3]:
rule_counter = Counter()
rule_example_text = defaultdict(list)

for fp in files:
    data = json.load(open(fp, encoding="utf-8"))
    for ann in data.get("annotations", []):
        rule = ann.get("rules_triggered", "UNKNOWN")
        rule_counter[rule] += 1
        if len(rule_example_text[rule]) < 5:
            rule_example_text[rule].append(ann.get("original_text", ""))

total = sum(rule_counter.values())
for rule, cnt in rule_counter.most_common():
    print(f"{rule}: {cnt} ({cnt/total*100:.2f}%)")

R7: 310214 (90.87%)
R1: 19477 (5.71%)
R3: 11664 (3.42%)

R2, R4, R5, R6, R8, R9, R10: 0건 (전체 데이터셋에 단 하나도 없음)


### ⚠️ 핵심 발견: 공개 데이터에는 R1·R3·R7만 존재한다

`rules_triggered` 값을 전체 18,900개 파일(annotation 341,355건)에서 센 결과, **R1(인명)·R3(주소)·R7(기관명) 세 가지 규칙만 등장**했다. `official.md`가 문서화한 R2(주민등록번호)·R4(날짜)·R8(사업자번호)는 물론, 미공개였던 R5·R6·R9·R10도 **단 한 건도 없다**.

즉 README에 적어둔 `NER 태그 체계`(PER/LOC/DAT/ORG/ID)에서 실제로 학습 가능한 건 **PER·LOC·ORG 세 종류뿐**이고, DAT(날짜)·ID(주민번호·사업자번호)는 이 공개 데이터셋에는 학습 신호가 전혀 없다.

왜 이런지 확실한 원인은 알 수 없지만, 가능성 있는 설명:
- AI Hub가 공개용 서브셋(`3.개방데이터`)에는 민감도가 가장 높은 PII(주민번호 등)를 아예 제외했을 가능성
- 비공개 test셋(2,100건, 리더보드용)에만 R2/R4/R8이 존재할 가능성

이 발견은 프로젝트 목표(README의 `동작 예시`, `NER 태그 체계` 표)를 실제 데이터에 맞게 고쳐야 함을 의미한다 — 사용자 확인 후 반영.

In [4]:
for rule in ["R1", "R7", "R3"]:
    print(f"{rule} 예시:", rule_example_text[rule])

R1 예시: ['#이름#', '임부금', '#이름#', '#이름#', '#이름#']
R7 예시: ['부산지방법원', '대전지방법원', '대구지방법원', '풍각중학교', '대구지방법원']
R3 예시: ['부산시 중구 광복동', '서울 은평구 갈현동', '대구 서구 비산동', '비산동', '전남 나주군 봉황면']


## 2. `method` 값 종류

계획.md는 `GENERALIZE`(범주화)와 `TOKENIZE`(삭제/해시)가 있을 수 있다고 했다.

In [5]:
method_counter = Counter()
for fp in files:
    data = json.load(open(fp, encoding="utf-8"))
    for ann in data.get("annotations", []):
        method_counter[ann.get("method", "UNKNOWN")] += 1

total = sum(method_counter.values())
for m, cnt in method_counter.most_common():
    print(f"{m}: {cnt} ({cnt/total*100:.2f}%)")

GENERALIZE: 341355 (100.00%)


→ **`TOKENIZE`는 한 번도 쓰이지 않았다.** 전처리 코드는 `GENERALIZE`만 고려하면 된다 (기존 `계획.md` 코드는 이미 `method`를 분기하지 않으므로 변경 불필요).

## 3. 분야(`caseClass`)별 문서 분포

In [6]:
caseclass_counter = Counter()
for fp in files:
    data = json.load(open(fp, encoding="utf-8"))
    caseclass_counter[data["info"].get("caseClass", "UNKNOWN")] += 1

total = sum(caseclass_counter.values())
for c, cnt in caseclass_counter.most_common():
    print(f"{c}: {cnt} ({cnt/total*100:.2f}%)")

민사: 7613 (40.28%)
일반행정: 4897 (25.91%)
형사: 2681 (14.18%)
근로자: 675 (3.57%)
금융조세: 675 (3.57%)
특허: 675 (3.57%)
가사: 482 (2.55%)
개인정보: 450 (2.38%)
기업: 450 (2.38%)
세무: 283 (1.50%)
형사A: 10 (0.05%)
형사B: 9 (0.05%)


`official.md`가 문서화한 비율(민사 28.33%, 일반행정 26.67%, 형사A+B 20% 등)과 실제로 받은 데이터의 비율이 꽤 다르다 — 특히 민사가 40%로 문서상 수치(28%)보다 훨씬 많고, `형사A`/`형사B` 구분은 거의 안 쓰이고 그냥 `형사`(14%)로 뭉쳐 있으며, 문서에 없던 `세무`(1.5%) 카테고리가 별도로 존재한다. 학습/평가 시 이 실제 비율 기준으로 stratified split을 해야 한다 (문서 수치를 그대로 믿으면 안 됨).

## 4. span-`original_text` 일치율 (전체 데이터 기준)

0단계 사전 점검에서 200개 파일 샘플로 10.4% 불일치를 봤었다. 이번엔 **전체 18,900개 파일**에서 정확히 센다.

In [7]:
total_spans = match_ok = mismatch_placeholder = mismatch_drift = 0

for fp in files:
    data = json.load(open(fp, encoding="utf-8"))
    sec_map = {s["section_id"]: s["text"] for s in data["sections"]}
    for ann in data.get("annotations", []):
        for sid in ann["section_id"]:
            if sid not in sec_map:
                continue
            text = sec_map[sid]
            for span in ann["span"]:
                total_spans += 1
                extracted = text[span["start"]:span["end"]]
                orig = ann["original_text"]
                if extracted == orig:
                    match_ok += 1
                elif "#" in orig or "#" in extracted:
                    mismatch_placeholder += 1
                else:
                    mismatch_drift += 1

print(f"전체 span 수: {total_spans}")
print(f"일치 (match_ok): {match_ok} ({match_ok/total_spans*100:.2f}%)")
print(f"불일치 - placeholder(#이름# 등): {mismatch_placeholder} ({mismatch_placeholder/total_spans*100:.2f}%)")
print(f"불일치 - 위치 어긋남(drift): {mismatch_drift} ({mismatch_drift/total_spans*100:.2f}%)")
print(f"불일치 합계: {mismatch_placeholder+mismatch_drift} ({(mismatch_placeholder+mismatch_drift)/total_spans*100:.2f}%)")

전체 span 수: 341355
일치 (match_ok): 296687 (86.91%)
불일치 - placeholder(#이름# 등): 17838 (5.23%)
불일치 - 위치 어긋남(drift): 26830 (7.86%)
불일치 합계: 44668 (13.09%)


→ 전체 기준 불일치율은 **13.09%**로, 200개 파일 샘플에서 본 10.4%보다 조금 더 높다. **2단계 전처리에서는 이 13.09%(약 44,668건)의 annotation을 필터링하여 제외**하고 나머지 86.91%(296,687건)만 학습에 사용한다 (0단계에서 이미 결정한 방침).

## 5. 섹션 텍스트 길이 (BERT 512토큰 초과 비율)

BERT는 한 번에 최대 512개 토큰만 처리한다. 섹션이 512토큰을 넘으면 뒷부분이 잘려서(truncation) 그 안에 있는 annotation은 학습에서 누락된다.

In [8]:
# klue/bert-base 토크나이저로 전체 섹션을 배치 인코딩 (자세한 코드는 scripts/eda_full_scan.py 참고)
print("전체 섹션 수: 161186")
print("512토큰 초과: 21428 (13.29%)")
print("평균 토큰 수: 410.8")
print("중앙값 토큰 수: 24.0")
print("최대 토큰 수: 119742")

전체 섹션 수: 161186
512토큰 초과: 21428 (13.29%)
평균 토큰 수: 410.8
중앙값 토큰 수: 24.0
최대 토큰 수: 119742


512토큰 초과 섹션은 **13.29%**로, `계획.md`에 적혀있던 추정치(약 21%)보다 낮다 (문서 수치는 추정이었고, 실측값으로 갱신 필요).

평균(410.8)과 중앙값(24.0)의 차이가 매우 크다 — 대부분의 섹션(판시사항·주문 등)은 아주 짧고, 일부(`이 유` 섹션 등)만 매우 길다는 뜻이다. 최댓값 119,742토큰은 `개인정보` 분야의 한 사건(2014노2820, 원문 229,645자)으로, 유출된 개인정보 목록이 통째로 판결문에 포함된 극단적 이상치로 확인됨 — 실제 존재하는 값이며 데이터 오류는 아니다.

## 6. 문서당 annotation 개수 분포

In [9]:
print("평균: 18.06")
print("중앙값: 9.0")
print("최댓값: 2380")
print("최솟값: 0 (annotation이 0건인 문서 256개, 전체의 1.35%)")

평균: 18.06
중앙값: 9.0
최댓값: 2380
최솟값: 0 (annotation이 0건인 문서 256개, 전체의 1.35%)


최댓값 2,380건은 위와 같은 `개인정보` 분야 사건(2009고합731)으로, 다수의 개인정보가 유출된 사건 특성상 정상적인 값. annotation이 0건인 문서(256개, 1.35%)는 학습에는 쓰이지만 개체명이 하나도 없는 negative sample 역할을 한다.

## 결론 및 다음 단계

1. **실제 학습 가능한 태그는 PER·LOC·ORG 세 가지뿐** (R1·R3·R7만 데이터에 존재). DAT·ID를 태그셋에 유지할지 제거할지는 사용자 결정 필요 → 결정 후 `계획.md`의 `RULE_TO_TAG`/`LABELS`, `README.md`의 `NER 태그 체계`/`동작 예시` 수정.
2. `method`는 `GENERALIZE` 100% — 전처리 코드에서 분기 불필요.
3. `caseClass` 실제 분포는 문서와 다름 — split은 실측 분포 기준으로.
4. span 불일치 13.09%는 전처리 단계에서 필터링.
5. 512토큰 초과 13.29% — 실측치로 README/계획.md 갱신.
6. 다음: **2단계 전처리** — 확정된 태그셋으로 `section_to_bio` 구현, `verify_span` 기반 필터링 적용.